# 01 — Preprocessing
Runs `clean_track` on all 90 trips (train + student_test) and writes `results/preprocessing_report.csv`.
Only the **track** is cleaned. Network and route are used for size statistics only, never as input to cleaning.

In [1]:
import sys, time
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
from config import RESULTS_DIR, GAP_THRESHOLD_S, PARTITIONS
from src.data.loader import iter_trips, load_trip, validate_trip
from src.data.preprocess import clean_track
from src.data.geo import haversine_m
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("GAP_THRESHOLD_S =", GAP_THRESHOLD_S)

GAP_THRESHOLD_S = 10.0


In [2]:
def route_length_m(trip):
    if trip["route"] is None:
        return np.nan
    a = trip["arcs"].loc[trip["route"]["arc_id"].to_numpy()]
    n = trip["nodes"]
    f, t = n.loc[a["from_node"].to_numpy()], n.loc[a["to_node"].to_numpy()]
    return float(haversine_m(f["lon"].to_numpy(), f["lat"].to_numpy(), t["lon"].to_numpy(), t["lat"].to_numpy()).sum())

rows, t0 = [], time.time()
for part, tid in iter_trips(PARTITIONS):
    trip = load_trip(part, tid)
    validate_trip(trip)
    clean, rep = clean_track(trip["track"])
    rows.append({
        "partition": part, "trip_id": tid,
        "raw_points": rep["raw_points"], "clean_points": rep["clean_points"],
        "dropped_invalid": rep["dropped_invalid"], "dropped_duplicates": rep["dropped_duplicates"],
        "out_of_order": rep["out_of_order"], "duplicate_timestamps": rep["duplicate_timestamps"],
        "n_gaps_over_threshold": rep["n_gaps_over_threshold"], "max_gap_s": rep["max_gap_s"],
        "n_segments": rep["n_segments"],
        "n_nodes": len(trip["nodes"]), "n_arcs": len(trip["arcs"]),
        "route_length": route_length_m(trip),      # metres, sum of haversine arc lengths
        "n_route_arcs": 0 if trip["route"] is None else len(trip["route"]),
    })
report = pd.DataFrame(rows)
report.to_csv(RESULTS_DIR / "preprocessing_report.csv", index=False)
print(f"{len(report)} trips in {time.time()-t0:.1f}s -> {RESULTS_DIR/'preprocessing_report.csv'}")
assert len(report) == 90
report.head()

90 trips in 1.8s -> /home/claude/pathsnap/results/preprocessing_report.csv


,partition,trip_id,raw_points,clean_points,dropped_invalid,dropped_duplicates,out_of_order,duplicate_timestamps,n_gaps_over_threshold,max_gap_s,n_segments,n_nodes,n_arcs,route_length,n_route_arcs
0,train,00000001,1315,1315,0,0,0,0,1,16.0,2,88809,165790,49854.610011,525
1,train,00000002,7243,7243,0,0,0,0,41,160.0,42,7936,16139,7963.185823,262
2,train,00000003,821,821,0,0,0,0,0,2.0,1,9138,18028,5004.546741,79
3,train,00000004,1064,1064,0,0,0,0,215,51.0,216,7350,15499,25546.780080,320
4,train,00000005,1045,1045,0,0,0,0,0,1.0,1,898,1579,30862.433330,105


## Summary per partition (min / median / max)

In [3]:
num = report.columns.drop(["partition", "trip_id"])
summary = report.groupby("partition")[list(num)].agg(["min", "median", "max"]).T
pd.set_option("display.width", 200)
summary

partition                     student_test          train
raw_points            min       262.000000     361.000000
                      median   1420.000000    1529.000000
                      max      3958.000000   19313.000000
clean_points          min       262.000000     361.000000
                      median   1420.000000    1529.000000
                      max      3958.000000   19313.000000
dropped_invalid       min         0.000000       0.000000
                      median      0.000000       0.000000
                      max         0.000000       0.000000
dropped_duplicates    min         0.000000       0.000000
                      median      0.000000       0.000000
                      max         0.000000       0.000000
out_of_order          min         0.000000       0.000000
                      median      0.000000       0.000000
                      max         0.000000       0.000000
duplicate_timestamps  min         0.000000       0.000000
                      median      0.000000       0.000000
                      max         0.000000       0.000000
n_gaps_over_threshold min         0.000000       0.000000
                      median      0.000000       0.500000
                      max        24.000000     215.000000
max_gap_s             min         1.000000       1.000000
                      median      2.000000      11.000000
                      max       210.000000  153064.000000
n_segments            min         1.000000       1.000000
                      median      1.000000       1.500000
                      max        25.000000     216.000000
n_nodes               min      1770.000000     299.000000
                      median   6596.500000    9608.000000
                      max     26635.000000  155003.000000
n_arcs                min      3577.000000     649.000000
                      median  13145.000000   19526.500000
                      max     56027.000000  307392.000000
route_length          min      5280.761185    5004.546741
                      median  14733.035978   14295.853123
                      max     64754.243115   94145.966802
n_route_arcs          min        95.000000      59.000000
                      median    250.000000     280.000000
                      max      1323.000000    2266.000000

## Totals per partition (issue counts summed over trips)

In [4]:
issues = ["raw_points","clean_points","dropped_invalid","dropped_duplicates","out_of_order",
          "duplicate_timestamps","n_gaps_over_threshold"]
totals = report.groupby("partition")[issues].sum()
totals.loc["ALL"] = totals.sum()
totals["trips_with_gaps"] = report.assign(g=report.n_gaps_over_threshold>0).groupby("partition")["g"].sum().reindex(totals.index[:-1]).tolist() + [int((report.n_gaps_over_threshold>0).sum())]
totals

,raw_points,clean_points,dropped_invalid,dropped_duplicates,out_of_order,duplicate_timestamps,n_gaps_over_threshold,trips_with_gaps
partition,,,,,,,,
student_test,19160,19160,0,0,0,0,27,2
train,208417,208417,0,0,0,0,590,40
ALL,227577,227577,0,0,0,0,617,42


## Top 5 trips by largest gap

In [5]:
top5 = report.nlargest(5, "max_gap_s")[["partition","trip_id","max_gap_s","n_gaps_over_threshold","n_segments","clean_points"]]
top5.assign(max_gap_h=(top5.max_gap_s/3600).round(2))

,partition,trip_id,max_gap_s,n_gaps_over_threshold,n_segments,clean_points,max_gap_h
72,train,00000090,153064.0,6,7,536,42.52
61,train,00000078,52810.0,25,26,2909,14.67
55,train,00000069,2658.0,2,3,2067,0.74
16,train,00000021,2161.0,2,3,15783,0.60
8,train,00000011,909.0,1,2,2828,0.25
